# Vector search with Milvus — `embeddings_tempo.jsonl`

End-to-end on the real embedding bundle: 1,734 ZTF objects, 192 dimensions, unit-normalised,
each carrying class probabilities and uncertainty estimates.

1. Load and inspect the embeddings
2. Connect to Milvus
3. Schema and index
4. Ingest and persist
5. Load into memory
6. Similarity search — query by example
7. Hybrid search — similarity *and* scalar filters
8. Evaluate: recall@k, and whether the neighbours mean anything
9. Hydrate full records
10. Reconnect and prove persistence

Defaults to Milvus Lite, which runs in-process — no server, no Docker. Flip one flag to point at a
real deployment.

## 0. Install

In [46]:
%pip install -q "pymilvus[milvus-lite]" numpy pandas

In [47]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from pymilvus import MilvusClient, DataType

pd.set_option("display.width", 200)
print("pymilvus", __import__("pymilvus").__version__)

pymilvus 3.0.0


## 1. Load the embeddings

Each line is one object. We are analyzing the embedding file here.
The fields:

| field | what it is |
|---|---|
| `object_id` | ZTF identifier — the join key back to the alert database |
| `embedding` | 192 floats, unit norm — the searchable vector |
| `dim`, `model_run`, `normalized` | provenance for the vector |
| `class_probs` | model's probability over `SNI`, `SNII`, `TDE`, `AGN`, `CV` |
| `uncertainty` | `vacuity`, `expected_entropy`, `mutual_information` |
| `n_events` | number of photometric detections behind the embedding |
| `horizon_days` | time span of the lightcurve fed to the model |

In [48]:
DATA_PATH = Path("embeddings_tempo.jsonl")
for candidate in [DATA_PATH, Path("/mnt/user-data/uploads/embeddings_tempo.jsonl")]:
    if candidate.exists():
        DATA_PATH = candidate
        break

records = [json.loads(line) for line in DATA_PATH.open()]
print(f"{len(records):,} records from {DATA_PATH}")
records[0] | {"embedding": f"<{len(records[0]['embedding'])} floats>"}

1,734 records from embeddings_tempo.jsonl


{'object_id': 'ZTF17aabtxcs',
 'model_run': 'model_bundle',
 'dim': 192,
 'n_events': 36,
 'horizon_days': 94.81260409997776,
 'normalized': True,
 'class_probs': {'SNI': 0.046507686376571655,
  'SNII': 0.05149657651782036,
  'TDE': 0.5132142305374146,
  'AGN': 0.04578394815325737,
  'CV': 0.3429974615573883},
 'uncertainty': {'vacuity': 0.22891685366630554,
  'expected_entropy': 1.0653109550476074,
  'mutual_information': 0.08068585395812988},
 'embedding': '<192 floats>'}

### Sanity checks before ingesting anything

Cheap, and they catch the failures that are miserable to debug later: a mixed-dimension file, a stray
`NaN`, duplicate IDs silently overwriting each other, or vectors that are not actually normalised when
you have chosen a metric that assumes they are.

In [49]:
DIM = records[0]["dim"]
embeddings = np.array([r["embedding"] for r in records], dtype=np.float32)
norms = np.linalg.norm(embeddings, axis=1)
object_ids = [r["object_id"] for r in records]

print("shape            ", embeddings.shape)
print("dims consistent  ", {r["dim"] for r in records}, all(len(r["embedding"]) == r["dim"] for r in records))
print("model runs       ", {r["model_run"] for r in records})
print("unique ids       ", len(set(object_ids)), "of", len(object_ids))
print("NaN / inf        ", bool(np.isnan(embeddings).any()), bool(np.isinf(embeddings).any()))
print("norms            ", f"{norms.min():.4f} .. {norms.max():.4f}")
print("value range      ", f"{embeddings.min():.4f} .. {embeddings.max():.4f}")

shape             (1734, 192)
dims consistent   {192} True
model runs        {'model_bundle'}
unique ids        1734 of 1734
NaN / inf         False False
norms             1.0000 .. 1.0000
value range       -0.2816 .. 0.3066


The vectors are already unit length, so **cosine and inner product give identical rankings** here.
The notebook uses `COSINE` because it stays correct even if a future bundle arrives un-normalised.

In [50]:
meta = pd.DataFrame({
    "row_id":     np.arange(len(records)),
    "object_id":  object_ids,
    "n_events":   [r["n_events"] for r in records],
    "horizon_days": [round(r["horizon_days"], 4) for r in records],
    "top_class":  [max(r["class_probs"], key=r["class_probs"].get) for r in records],
    "top_prob":   [round(max(r["class_probs"].values()), 5) for r in records],
    "vacuity":    [round(r["uncertainty"]["vacuity"], 5) for r in records],
    "entropy":    [round(r["uncertainty"]["expected_entropy"], 5) for r in records],
    "mutual_info":[round(r["uncertainty"]["mutual_information"], 5) for r in records],
})

print(meta["top_class"].value_counts().to_string())
print()
meta.describe().T[["min", "50%", "max"]]

top_class
SNI     694
AGN     599
SNII    279
TDE     109
CV       53



,min,50%,max
row_id,0.00000,866.500000,1733.00000
n_events,1.00000,58.000000,1360.00000
horizon_days,0.00480,53.932400,100.00000
top_prob,0.29381,0.771940,0.87365
vacuity,0.15309,0.191025,0.47216
entropy,0.50128,0.759910,1.37948
mutual_info,0.05326,0.066530,0.16711


`top_class` is the model's own argmax, not an external label. That distinction matters in section 8 —
keep it in mind.

## 2. Connect

Milvus Lite takes a file path and runs inside this process. A real deployment takes `grpc://host:port`.
Same API either way, so nothing below this cell changes.

In [51]:
USE_REMOTE = False

LOCAL_URI  = "./milvus_tempo.db"
REMOTE_URI = "grpc://milvus.example.org:50051"
TOKEN      = ""                          # "user:password" if auth is enabled

def connect():
    return MilvusClient(uri=REMOTE_URI, token=TOKEN) if USE_REMOTE else MilvusClient(uri=LOCAL_URI)

client = connect()
COLLECTION = "tempo_embeddings"
print("connected:", REMOTE_URI if USE_REMOTE else LOCAL_URI)
print("collections:", client.list_collections())

connected: ./milvus_tempo.db
collections: ['tempo_embeddings']


If a remote connection hangs rather than erroring, that is almost always a blocked port rather than a
dead server. Check reachability from the machine running this notebook:

```bash
nc -zv milvus.example.org 50051
```

## 3. Schema and index

Two decisions get baked in here.

**Schema.** The vector plus the scalars you intend to filter on. Those scalars sit *alongside* the
vector — they are not encoded inside it. Milvus has no field for `class_probs` or `uncertainty` as
nested objects, so the useful pieces are flattened out and the full record stays in the source database.

**Index.** The metric is needed at *build* time, not just query time — constructing a neighbour graph
means measuring closeness, so it becomes part of the structure. Build and search must agree.

At 1,734 rows `FLAT` would be instant and exact; `HNSW` is used here because it is what you would run at
production scale, and the code should be the same either way.

In [53]:
if client.has_collection(COLLECTION):
    client.drop_collection(COLLECTION)

schema = client.create_schema(auto_id=False, enable_dynamic_field=False)
schema.add_field("row_id",       DataType.INT64,        is_primary=True)
schema.add_field("embedding",    DataType.FLOAT_VECTOR, dim=DIM)
schema.add_field("object_id",    DataType.VARCHAR,      max_length=32)
schema.add_field("top_class",    DataType.VARCHAR,      max_length=16)
schema.add_field("top_prob",     DataType.FLOAT)
schema.add_field("n_events",     DataType.INT64)
schema.add_field("horizon_days", DataType.FLOAT)
schema.add_field("vacuity",      DataType.FLOAT)
schema.add_field("entropy",      DataType.FLOAT)
schema.add_field("mutual_info",  DataType.FLOAT)

index_params = client.prepare_index_params()
index_params.add_index(
    field_name="embedding",
    index_type="HNSW",
    metric_type="COSINE",
    params={"M": 16, "efConstruction": 200},
)
# Scalar indexes for the fields used in filters. Milvus Lite supports INVERTED;
# a full deployment also offers STL_SORT for numeric ranges.
index_params.add_index(field_name="top_class", index_type="INVERTED")
index_params.add_index(field_name="n_events",  index_type="INVERTED")

client.create_collection(COLLECTION, schema=schema, index_params=index_params)
print("created:", COLLECTION, "| indexes:", client.list_indexes(COLLECTION))
COLLECTION.

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1232, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!
ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1232, in AllocTimestamp
    raise NotImplementedError('Method no

created: tempo_embeddings | indexes: ['embedding', 'top_class', 'n_events']


## 4. Ingest and persist

An insert lands in a durable write log, is consumed into an in-memory *growing* segment, and once that
segment hits a size or time threshold it is **sealed**, flushed to object storage, and indexed.

`flush()` forces sealing now instead of waiting. Useful after a bulk load like this one — but do not call
it after every small insert in production, or you create a swarm of tiny segments that compaction then
has to clean up.

In [54]:
BATCH = 500

for start in range(0, len(meta), BATCH):
    chunk = meta.iloc[start:start + BATCH]
    rows = [{
        "row_id":       int(r.row_id),
        "embedding":    embeddings[r.row_id],
        "object_id":    r.object_id,
        "top_class":    r.top_class,
        "top_prob":     float(r.top_prob),
        "n_events":     int(r.n_events),
        "horizon_days": float(r.horizon_days),
        "vacuity":      float(r.vacuity),
        "entropy":      float(r.entropy),
        "mutual_info":  float(r.mutual_info),
    } for r in chunk.itertuples()]
    client.insert(COLLECTION, rows)

client.flush(COLLECTION)
print("row count:", client.get_collection_stats(COLLECTION)["row_count"])

row count: 1734


## 5. Load

The step that catches people out coming from a document database: **data must be loaded into memory
before it can be searched.** Persisted is not searchable. Loading lifts sealed segments and their indexes
off storage into query-node RAM, which is where searches actually run.

In Milvus Lite, the local (.db) file plays the role of the object store, and loading makes its contents available to RAM.

In [55]:
client.load_collection(COLLECTION)
print(client.get_load_state(COLLECTION))
# client.release_collection(COLLECTION)   # frees the RAM again, keeps the data

{'state': <LoadState: Loaded>}


## 6. Similarity search — query by example

The thing a scalar query cannot do. You are not naming fields or setting thresholds; you hand over an
object and ask what resembles it. The model decides what "resemble" means.

With `COSINE` the returned distance is a similarity in `[-1, 1]` and **higher is closer**. With `L2` it
is a distance and lower is closer. Do not mix them up when sorting or thresholding.

In [56]:
def search(vector, limit=10, filter_expr="", ef=64, fields=("object_id", "top_class", "top_prob", "n_events")):
    hits = client.search(
        COLLECTION, data=[vector], limit=limit, filter=filter_expr,
        output_fields=list(fields),
        search_params={"metric_type": "COSINE", "params": {"ef": ef}},
    )[0]
    return pd.DataFrame([
        {"row_id": h["row_id"], "similarity": round(h["distance"], 4), **h["entity"]}
        for h in hits
    ])

# Start with a confidently-classified object so the mechanism is visible before the
# hard cases. Section 8 deals with the ambiguous ones.
QUERY_ROW = int(meta.query("n_events > 50")["top_prob"].idxmax())
q = meta.iloc[QUERY_ROW]
print(f"query: {q.object_id}  |  model says {q.top_class} at p={q.top_prob:.3f}  |  {q.n_events} events")

search(embeddings[QUERY_ROW], limit=10)

query: ZTF20aaoyovs  |  model says AGN at p=0.869  |  51 events


,row_id,similarity,object_id,top_class,top_prob,n_events
0,333,1.0000,ZTF20aaoyovs,AGN,0.86886,51
1,359,0.9954,ZTF20abudlns,AGN,0.86648,70
2,360,0.9934,ZTF20accxjxh,AGN,0.86888,37
3,182,0.9918,ZTF19aaapmnk,AGN,0.86720,30
4,315,0.9918,ZTF20aacxmdk,AGN,0.87134,22
5,473,0.9915,ZTF26aacqqjz,AGN,0.86958,34
6,311,0.9906,ZTF20aaawejz,AGN,0.86791,51
7,476,0.9902,ZTF26aafxyji,AGN,0.86758,24
8,308,0.9900,ZTF19adcgcdf,AGN,0.86056,173
9,679,0.9888,ZTF19abcgdoz,AGN,0.87085,11


The self-match at similarity 1.0 is the query object finding itself — expected, and worth excluding when
you evaluate anything. Note how tight the rest are: this embedding space has genuine structure.

## 7. Hybrid search — similarity *and* filters

Neither approach suffices alone. Scalar filters need you to know what to ask for; similarity search alone
ignores the constraints you actually have — well-sampled lightcurves, confident predictions, low model
uncertainty. Combine them in one query.

A good engine applies the filter *during* graph traversal: each candidate is checked against the
predicates and only collected if it passes, while failing nodes can still be traversed *through* to reach
others, protecting recall. Filtering first can fragment the graph; filtering afterwards can leave you
holding nothing.

Filter syntax is a small expression language: comparisons, `and` / `or` / `not`, `in [...]`,
`like "ZTF17%"`.

In [58]:
FILTER = "n_events > 50 and top_prob > 0.7 and vacuity < 0.20"
print("filter:", FILTER, "\n")
search(embeddings[QUERY_ROW], limit=10, filter_expr=FILTER,
       fields=("object_id", "top_class", "top_prob", "n_events", "vacuity"))

filter: n_events > 50 and top_prob > 0.7 and vacuity < 0.20 



,row_id,similarity,object_id,top_class,top_prob,n_events,vacuity
0,333,1.0000,ZTF20aaoyovs,AGN,0.86886,51,0.16341
1,359,0.9954,ZTF20abudlns,AGN,0.86648,70,0.16618
2,311,0.9906,ZTF20aaawejz,AGN,0.86791,51,0.16423
3,308,0.9900,ZTF19adcgcdf,AGN,0.86056,173,0.17039
4,412,0.9870,ZTF23aaecgge,AGN,0.86154,189,0.17178
5,471,0.9860,ZTF26aabdijd,AGN,0.86721,63,0.16535
6,464,0.9842,ZTF25acgpzzv,AGN,0.86878,51,0.16348
7,242,0.9842,ZTF19aaoznlz,AGN,0.86863,51,0.16292
8,303,0.9838,ZTF19adcdfyb,AGN,0.86022,206,0.17164
9,77,0.9808,ZTF18abodiov,AGN,0.85913,524,0.17283


If a filter is aggressive enough to exclude everything genuinely similar, the remaining results can come
back with *negative* similarity — actively unlike the query. That is not an error; it is the search
truthfully reporting that nothing eligible resembles what you asked for. Treat a negative or near-zero
top similarity as "no real match under this filter" rather than as a ranked answer.

In [ ]:
# How much does each predicate cost you? Useful before you bake one into a pipeline.
for expr in ["",
             "n_events > 50",
             "top_prob > 0.7",
             "vacuity < 0.20",
             "top_class == 'TDE'",
             FILTER]:
    n = client.query(COLLECTION, filter=expr or "row_id >= 0", output_fields=["count(*)"])[0]["count(*)"]
    print(f"{n:>5,} of {len(meta):,}   {expr or '(no filter)'}")

A restrictive filter shrinks the eligible pool, so the graph walk has to wander further to fill `limit`
results. Widening `ef` is the usual remedy — it costs latency and buys back recall. `ef` is the size of the candidate list the search keeps alive while walking the graph

> **Milvus Lite caveat.** Lite ignores search-time tuning parameters, so `ef` will not visibly move
> latency or recall in this notebook. The code is correct and the knob is real; point `USE_REMOTE` at a
> standalone or distributed deployment to see the curve. Do not conclude from a flat line here that the
> knob does nothing.

---

### What Milvus Lite hides

Lite runs everything in-process. A few things it papers over are load-bearing on a cluster:

- **Load is mandatory.** Searching an unloaded collection fails outright.
- **Inserts are not instantly visible.** New vectors must propagate before query nodes see them.
  `consistency_level` is the dial — `Strong` waits for the newest write, `Bounded` / `Eventually` return
  faster and may miss it.
- **Search-time parameters are honoured.** `ef` and `nprobe` genuinely trade latency against recall.
- **Segments, compaction, and node counts are real.** Each sealed segment carries its own index; a search
  fans out across all of them and merges. Compaction keeps that fan-out manageable.
- **Not every index type is available in Lite** — quantised indexes in particular are a cluster feature.

The code is identical across all three deployment modes, which is the point of developing against Lite.